In [49]:
from clearml import Task, Dataset
import pandas as pd
import joblib
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

In [ ]:
task = Task.init(
    project_name="Course Work",
    task_name="Model Logistic Regression"
)

ClearML Task: created new task id=e787958df6ef49ac8c6f3c3f50f6b782


Could not fetch GPU stats: NVML Shared Library Not Found


ClearML results page: https://app.clear.ml/projects/61047c7a66aa40eab8cd971dc2d8b828/experiments/e787958df6ef49ac8c6f3c3f50f6b782/output/log


ClearML Monitor: GPU monitoring failed getting GPU reading, switching off GPU monitoring


In [51]:
dataset = Dataset.get(
    dataset_name="Transactions Features",
    dataset_project="Course Work"
)

data_path = dataset.get_local_copy()

train = pd.read_csv(f"{data_path}/train.csv")
test = pd.read_csv(f"{data_path}/test.csv")

In [52]:
dataset = Dataset.get(
    dataset_name="Transactions Features",
    dataset_project="Course Work"
)

data_path = dataset.get_local_copy()

train = pd.read_csv(f"{data_path}/train.csv")
test = pd.read_csv(f"{data_path}/test.csv")

In [53]:
# Подготовка признаков
feature_cols = ["recency"]
X_train = train[feature_cols]
y_train = train["target"]
X_test = test[feature_cols]
y_test = test["target"]

In [54]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [55]:
params = {
    "C": 1.0,
    "max_iter": 1000,
    "solver": "lbfgs",
    "random_state": 42
}
task.connect(params)

{'C': 1.0, 'max_iter': 1000, 'solver': 'lbfgs', 'random_state': 42}

In [56]:
model = LogisticRegression(**params)
model.fit(X_train_scaled, y_train)

LogisticRegression(max_iter=1000, random_state=42)

In [57]:
y_train_proba = model.predict_proba(X_train_scaled)[:, 1]
y_test_proba  = model.predict_proba(X_test_scaled)[:, 1]

In [58]:
auc_train = roc_auc_score(y_train, y_train_proba)
auc_test  = roc_auc_score(y_test, y_test_proba)

print(f"ROC-AUC (train): {auc_train:.4f}")
print(f"ROC-AUC (test) : {auc_test:.4f}")

# Логирование в ClearML
task.get_logger().report_scalar("ROC-AUC", "train", auc_train, 0)
task.get_logger().report_scalar("ROC-AUC", "test",  auc_test, 0)

ROC-AUC (train): 0.8207
ROC-AUC (test) : 0.8135


In [59]:
task.get_logger().report_scalar(
    title="ROC-AUC", 
    series="test", 
    value=auc_roc, 
    iteration=0
)

In [60]:
joblib.dump(model, "model_logistic.pkl")
task.upload_artifact("model_logistic", "model_logistic.pkl")

print("Эксперимент завершен")
task.close()

Эксперимент завершен
